In [ ]:
# | default_exp parser.utils


# Parser Utils
> General-purpose text normalization, slug utilities, and file helpers.

In [ ]:
#| export
import re
from pathlib import Path
from seootter.parser.metadata import extract_notebook_content, remove_metadata

SPECIAL_PREFIXES = ("#", "mailto:", "tel:", "javascript:")


## Text Utilities

General-purpose text normalization and analysis helpers.

In [ ]:
# | export
def normalize_text(text: str  # Text to normalize
                   ) -> str:
    "Normalize text by collapsing extra whitespace."
    return re.sub(r"\s+", " ", text).strip()


In [ ]:
# | export
def detect_phone_numbers(text: str  # Text to search
                         ) -> list[str]:
    "Extract phone numbers from text."
    phone_regex = re.compile(r"(\+\d{1,3})?\s*?(\d{3})\s*?(\d{3})\s*?(\d{3,4})")
    return ["".join(g) for g in phone_regex.findall(text)]


In [ ]:
# | test
from fastcore.test import test_eq
from pathlib import Path

sample_dir = Path("sample")
if not sample_dir.exists():
    sample_dir = Path("../sample")
with open(sample_dir / "example.md") as file:
    content = file.read()
phones = detect_phone_numbers(content)
test_eq("+966503139675" in phones, True)


In [ ]:
# | export
def calculate_similarity(text1: str,  # First text
                         text2: str  # Second text
                         ) -> float:
    "Calculate similarity ratio between two texts using SequenceMatcher."
    from difflib import SequenceMatcher
    return SequenceMatcher(None, text1, text2).ratio()


In [ ]:
# | export
def is_special_url(url: str  # URL to check
                   ) -> bool:
    "Check if URL is an anchor, mailto, tel, or javascript link."
    return any(url.startswith(p) for p in SPECIAL_PREFIXES)


## File Utilities

Helpers for finding and reading markdown and notebook files.

In [ ]:
# | export
def get_file_paths(pattern: str  # Glob pattern (e.g. '**/*.md')
                   ) -> list[str]:
    "Get file paths matching a glob pattern."
    import glob
    return glob.glob(pattern, recursive=True)


In [ ]:
# | export
def get_file_name(file_path: str  # Path to file
                  ) -> str:
    "Extract filename without extension from a path."
    return Path(file_path).stem


In [ ]:
# | export
def get_markdown_files(directory: str  # Directory to search
                       ) -> list[str]:
    "Get all markdown filenames (without extension) from a directory."
    import os
    return [f.replace(".md", "") for f in os.listdir(directory)
            if f.endswith(".md") and f != ".obsidian"]

def get_page_content(file_path: str,  # Path to markdown or notebook file
                     is_quarto: bool = False  # Whether the notebook is a Quarto doc
                     ) -> str:
    "Read a file and return its text content, stripping frontmatter."
    with open(file_path, "r") as f: raw = f.read()
    if file_path.endswith(".ipynb"): return extract_notebook_content(raw, is_quarto=is_quarto)
    return remove_metadata(raw)


## Arabic Slug Utilities

Helpers for converting Arabic filenames to URL-friendly slugs.

In [ ]:
# | export
def arabic_to_slug(text: str  # Arabic text to convert
                   ) -> str:
    "Convert Arabic text to a URL-friendly slug."
    char_map = {"ا": "a", "ب": "b", "ت": "t", "ث": "th", "ج": "j", "ح": "h", "خ": "kh", "د": "d",
                "ذ": "th", "ر": "r", "ز": "z", "س": "s", "ش": "sh", "ص": "s", "ض": "d", "ط": "t",
                "ظ": "z", "ع": "", "غ": "gh", "ف": "f", "ق": "q", "ك": "k", "ل": "l", "م": "m",
                "ن": "n", "ه": "h", "و": "w", "ي": "y", "ة": "h", " ": "-"}
    slug = "".join(char_map.get(c, c) for c in text.strip().lower())
    while "--" in slug: slug = slug.replace("--", "-")
    return slug.strip("-")

In [ ]:
# | export
def map_files_to_slugs(directory: str  # Directory containing Arabic markdown files
                       ) -> dict[str, str]:
    "Map markdown filenames to their URL slugs."
    return {f: arabic_to_slug(f) for f in get_markdown_files(directory)}


## HTML Fetching

In [ ]:
#| export
def fetch_url_as_markdown(url: str,  # Live URL to fetch
                          extractor: str = "trafilatura"  # 'trafilatura' or 'html2text'
                          ) -> str:
    "Fetch a live URL and return its main content as markdown."
    import httpx, lxml.html
    from lxml.html.clean import Cleaner

    body = lxml.html.fromstring(httpx.get(url).text).xpath('//body')[0]
    body = Cleaner(javascript=True, style=True).clean_html(body)
    cts = ''.join(lxml.html.tostring(c, encoding='unicode') for c in body)

    if extractor == "trafilatura":
        from trafilatura import extract
        if '<article>' not in cts.lower(): cts = f'<article>{cts}</article>'
        return extract(f'<html><body>{cts}</body></html>', output_format='markdown',
                       favor_recall=True, include_tables=True,
                       include_links=False, include_images=False) or ""
    else:
        from html2text import HTML2Text
        h = HTML2Text(bodywidth=5000)
        h.ignore_links = True
        h.ignore_images = True
        return h.handle(cts).strip()
